# Stage 2: Multi-Head Attention

## Why One Head Isn't Enough

In the previous notebook, we built **self-attention** - a powerful mechanism that lets tokens attend to each other. But there's a limitation:

**Single attention can only learn one "type" of relationship at a time.**

Consider the sentence: *"The cat sat on the mat because it was tired."*

The word "it" needs to attend to:
- **Syntactic relationship**: "it" → "cat" (pronoun reference)
- **Positional relationship**: nearby words for context
- **Semantic relationship**: "tired" → relates to living things

**Multi-Head Attention solves this** by running multiple attention operations in parallel, each learning different relationship patterns!

### The Formula

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

where each head is:
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

## Visual Architecture

```
Input X (batch, seq_len, d_model=512)
    │
    ├──────────────┬──────────────┬──────────────┐
    ▼              ▼              ▼              ▼
 Head 1         Head 2         Head 3    ...  Head 8
 (d_k=64)       (d_k=64)       (d_k=64)       (d_k=64)
    │              │              │              │
    ▼              ▼              ▼              ▼
 Attention     Attention     Attention     Attention
  Output        Output        Output        Output
 (64-dim)      (64-dim)      (64-dim)      (64-dim)
    │              │              │              │
    └──────────────┴──────────────┴──────────────┘
                          │
                          ▼
                   Concatenate
                (batch, seq_len, 512)
                          │
                          ▼
                   W^O projection
                          │
                          ▼
               Output (batch, seq_len, d_model=512)
```

**Key insight**: Instead of one 512-dim attention, we use 8 parallel 64-dim attentions.
Same total computation, but **more expressive**!

---
## Step 1: Single Head Attention (Review)

Let's quickly recap our single-head attention from before:

In [4]:
def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        query: (batch, heads, seq_len, d_k)
        key:   (batch, heads, seq_len, d_k)
        value: (batch, heads, seq_len, d_v)
        mask:  optional mask
        dropout: optional dropout layer
    
    Returns:
        output: weighted sum of values
        attention_weights: the attention scores
    """
    d_k = query.size(-1)
    
    # Attention scores: (batch, heads, seq_len, seq_len)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attention_weights = F.softmax(scores, dim=-1)
    
    if dropout is not None:
        attention_weights = dropout(attention_weights)
    
    output = torch.matmul(attention_weights, value)
    
    return output, attention_weights

---
## Step 2: Multi-Head Attention Implementation

Now let's build Multi-Head Attention from scratch!

### The Trick: Efficient Parallel Computation

Instead of running 8 separate attention operations, we can:
1. Project once to get all Q, K, V for all heads
2. Reshape to separate heads
3. Do one batched attention operation
4. Reshape back and project

In [5]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention as described in "Attention Is All You Need"
    
    Instead of performing a single attention with d_model dimensions,
    we split into h heads, each with d_k = d_model/h dimensions.
    This allows the model to attend to information at different positions
    from different representation subspaces.
    """
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Args:
            d_model: Model dimension (must be divisible by num_heads)
            num_heads: Number of attention heads
            dropout: Dropout probability
        """
        super().__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        
        # Linear projections for Q, K, V (all heads at once)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        
        # Output projection
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: (batch, seq_len, d_model)
            key:   (batch, seq_len, d_model)
            value: (batch, seq_len, d_model)
            mask:  optional mask (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len)
        
        Returns:
            output: (batch, seq_len, d_model)
            attention_weights: (batch, num_heads, seq_len, seq_len)
        """
        batch_size = query.size(0)
        seq_len = query.size(1)
        
        # Step 1: Linear projections
        # Shape: (batch, seq_len, d_model)
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)
        
        # Step 2: Split into multiple heads
        # Reshape: (batch, seq_len, d_model) → (batch, seq_len, num_heads, d_k)
        # Transpose: → (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Step 3: Apply scaled dot-product attention
        # attention_output: (batch, num_heads, seq_len, d_k)
        # attention_weights: (batch, num_heads, seq_len, seq_len)
        attention_output, attention_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )
        
        # Step 4: Concatenate heads
        # Transpose: (batch, num_heads, seq_len, d_k) → (batch, seq_len, num_heads, d_k)
        # Reshape: → (batch, seq_len, d_model)
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len, self.d_model)
        
        # Step 5: Final linear projection
        output = self.W_O(attention_output)
        
        return output, attention_weights

In [6]:
# Test our implementation
d_model = 512
num_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# Create random input
x = torch.randn(batch_size, seq_len, d_model)

# Forward pass (self-attention: query=key=value)
output, attn_weights = mha(x, x, x)

print(f"Input shape:            {x.shape}")
print(f"Output shape:           {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nDimension per head (d_k): {d_model // num_heads}")
print(f"Number of parameters: {sum(p.numel() for p in mha.parameters()):,}")

Input shape:            torch.Size([2, 10, 512])
Output shape:           torch.Size([2, 10, 512])
Attention weights shape: torch.Size([2, 8, 10, 10])

Dimension per head (d_k): 64
Number of parameters: 1,048,576


---
## Step 3: Understanding the Shape Transformations

The key insight is how we reshape tensors to process all heads in parallel:

In [7]:
# Let's trace through the shapes step by step
batch_size, seq_len, d_model = 2, 6, 512
num_heads = 8
d_k = d_model // num_heads  # 64

# Input
x = torch.randn(batch_size, seq_len, d_model)
print(f"1. Input: {x.shape}")
print(f"   → (batch=2, seq_len=6, d_model=512)")

# After linear projection (still same shape)
W_Q = nn.Linear(d_model, d_model, bias=False)
Q = W_Q(x)
print(f"\n2. After projection: {Q.shape}")
print(f"   → Still (batch=2, seq_len=6, d_model=512)")

# Reshape to separate heads
Q_reshaped = Q.view(batch_size, seq_len, num_heads, d_k)
print(f"\n3. After view(batch, seq_len, num_heads, d_k): {Q_reshaped.shape}")
print(f"   → Now we can see the 8 heads, each with 64 dimensions")

# Transpose to put heads before sequence
Q_transposed = Q_reshaped.transpose(1, 2)
print(f"\n4. After transpose(1, 2): {Q_transposed.shape}")
print(f"   → (batch=2, num_heads=8, seq_len=6, d_k=64)")
print(f"   → Now each head can process the sequence independently!")

1. Input: torch.Size([2, 6, 512])
   → (batch=2, seq_len=6, d_model=512)

2. After projection: torch.Size([2, 6, 512])
   → Still (batch=2, seq_len=6, d_model=512)

3. After view(batch, seq_len, num_heads, d_k): torch.Size([2, 6, 8, 64])
   → Now we can see the 8 heads, each with 64 dimensions

4. After transpose(1, 2): torch.Size([2, 8, 6, 64])
   → (batch=2, num_heads=8, seq_len=6, d_k=64)
   → Now each head can process the sequence independently!


### Why This Shape?

```
(batch, num_heads, seq_len, d_k)
   ↓        ↓         ↓       ↓
   2        8         6      64
```

- **batch=2**: We process 2 sequences in parallel
- **num_heads=8**: Each sequence has 8 independent attention computations
- **seq_len=6**: Each head looks at 6 tokens
- **d_k=64**: Each token is represented by 64 dimensions per head

When we compute `Q @ K^T`:
- Shape: `(2, 8, 6, 64) @ (2, 8, 64, 6) = (2, 8, 6, 6)`
- Result: 8 attention matrices, each 6×6, for each of 2 batches

---
## Step 4: Visualizing Multiple Heads

Let's see how different heads learn different attention patterns:

In [ ]:
# Simulate a sentence with actual tokens
tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
d_model = 64
num_heads = 4

# Create embeddings (random for demonstration)
torch.manual_seed(42)
x = torch.randn(1, seq_len, d_model)

# Multi-head attention
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)

# Forward pass
output, attn_weights = mha(x, x, x)

print(f"Attention weights shape: {attn_weights.shape}")
print(f"→ (batch=1, heads={num_heads}, seq_len={seq_len}, seq_len={seq_len})")

In [ ]:
# Visualize all heads
fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))

for head_idx in range(num_heads):
    ax = axes[head_idx]
    attn = attn_weights[0, head_idx].detach().numpy()
    
    im = ax.imshow(attn, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {head_idx + 1}', fontsize=12)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, fontsize=9, rotation=45)
    ax.set_yticklabels(tokens, fontsize=9)
    
    if head_idx == 0:
        ax.set_ylabel('Query (attending FROM)', fontsize=10)
    if head_idx == num_heads // 2:
        ax.set_xlabel('Key (attending TO)', fontsize=10)

# Add colorbar
plt.colorbar(im, ax=axes, shrink=0.8, label='Attention Weight')

plt.suptitle('Multi-Head Attention: Each Head Learns Different Patterns', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### What Different Heads Might Learn

In trained models, different heads often specialize:

| Head Type | Pattern | Example |
|-----------|---------|--------|
| **Positional** | Attend to adjacent tokens | "sat" → "cat", "on" |
| **Syntactic** | Attend to grammatical relations | "mat" → "the" (determiner) |
| **Semantic** | Attend to related meanings | "cat" → "sat" (animate + action) |
| **Coreference** | Attend to referred entities | "it" → "cat" |

This is why multi-head attention is so powerful - it captures multiple types of relationships simultaneously!

---
## Step 5: Multi-Head Attention with Causal Masking

For autoregressive models (GPT-style), we need causal masking. Let's add that:

In [ ]:
def create_causal_mask(seq_len):
    """
    Create a causal mask for autoregressive attention.
    Shape: (1, 1, seq_len, seq_len) for broadcasting
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

# Test with causal mask
tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)

causal_mask = create_causal_mask(seq_len)
print("Causal mask shape:", causal_mask.shape)
print("\nCausal mask (1=attend, 0=block):")
print(causal_mask[0, 0].numpy().astype(int))

In [ ]:
# Apply causal mask to multi-head attention
x = torch.randn(1, seq_len, d_model)
output_causal, attn_weights_causal = mha(x, x, x, mask=causal_mask)

# Visualize causal attention for all heads
fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))

for head_idx in range(num_heads):
    ax = axes[head_idx]
    attn = attn_weights_causal[0, head_idx].detach().numpy()
    
    im = ax.imshow(attn, cmap='Oranges', vmin=0, vmax=1)
    ax.set_title(f'Head {head_idx + 1} (Causal)', fontsize=12)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, fontsize=9, rotation=45)
    ax.set_yticklabels(tokens, fontsize=9)

plt.colorbar(im, ax=axes, shrink=0.8, label='Attention Weight')
plt.suptitle('Causal Multi-Head Attention: Each Token Only Sees Past', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Step 6: Parameter Count Analysis

Understanding where the parameters are is crucial for model efficiency:

In [ ]:
d_model = 512
num_heads = 8

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

print("Parameter breakdown:")
print("=" * 40)

total = 0
for name, param in mha.named_parameters():
    count = param.numel()
    total += count
    print(f"{name:10} → {param.shape} = {count:,} params")

print("=" * 40)
print(f"Total: {total:,} parameters")
print(f"\nBreakdown:")
print(f"  W_Q, W_K, W_V: 3 × {d_model}² = {3 * d_model**2:,}")
print(f"  W_O:          1 × {d_model}² = {d_model**2:,}")
print(f"  Total:        4 × {d_model}² = {4 * d_model**2:,}")

### Key Insight: Parameters Don't Increase With More Heads!

Whether we use 1 head or 16 heads, the parameter count stays the same: **4 × d_model²**

This is because:
- Each head has smaller projections: `d_model → d_k` where `d_k = d_model / num_heads`
- More heads × smaller projections = same total parameters

**Multi-head attention gives us more expressiveness for free!**

---
## Step 7: Comparison - Single Head vs Multi-Head

Let's compare the expressiveness:

In [ ]:
# Compare single-head (512-dim) vs multi-head (8 × 64-dim)
d_model = 512
seq_len = 8
batch_size = 1

# Single head attention (simulated by using num_heads=1)
single_head = MultiHeadAttention(d_model=d_model, num_heads=1, dropout=0.0)

# Multi-head attention
multi_head = MultiHeadAttention(d_model=d_model, num_heads=8, dropout=0.0)

# Same input
torch.manual_seed(123)
x = torch.randn(batch_size, seq_len, d_model)

# Forward pass
_, single_attn = single_head(x, x, x)
_, multi_attn = multi_head(x, x, x)

print(f"Single-head attention shape: {single_attn.shape}")
print(f"Multi-head attention shape:  {multi_attn.shape}")

In [ ]:
# Visualize the difference
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Single head (show in first row, centered)
for i in range(5):
    if i == 2:  # Center position
        axes[0, i].imshow(single_attn[0, 0].detach().numpy(), cmap='Greens')
        axes[0, i].set_title('Single Head (1 × 512-dim)', fontsize=10)
        axes[0, i].set_xlabel('Key')
        axes[0, i].set_ylabel('Query')
    else:
        axes[0, i].axis('off')

# Multi-head (show first 5 heads in second row)
for i in range(5):
    if i < 5:
        axes[1, i].imshow(multi_attn[0, i].detach().numpy(), cmap='Blues')
        axes[1, i].set_title(f'Head {i+1} (8 × 64-dim)', fontsize=10)
        if i == 0:
            axes[1, i].set_ylabel('Query')
        if i == 2:
            axes[1, i].set_xlabel('Key')

plt.suptitle('Single Head vs Multi-Head: Same Parameters, More Patterns!', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 8: Real-World Implementation Details

Let's look at some practical improvements used in modern transformers:

### 8.1 Attention with Bias (Optional)

In [ ]:
class MultiHeadAttentionWithBias(nn.Module):
    """
    Multi-Head Attention with optional bias terms.
    
    Some models (like BERT) use bias, others (like LLaMA) don't.
    No bias → fewer parameters, sometimes better for training stability.
    """
    
    def __init__(self, d_model, num_heads, dropout=0.1, bias=True):
        super().__init__()
        
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Option to include or exclude bias
        self.W_Q = nn.Linear(d_model, d_model, bias=bias)
        self.W_K = nn.Linear(d_model, d_model, bias=bias)
        self.W_V = nn.Linear(d_model, d_model, bias=bias)
        self.W_O = nn.Linear(d_model, d_model, bias=bias)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        seq_len = query.size(1)
        
        Q = self.W_Q(query).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(key).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(value).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        attention_output, attention_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )
        
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len, self.d_model)
        
        return self.W_O(attention_output), attention_weights

# Compare parameter counts
mha_no_bias = MultiHeadAttentionWithBias(512, 8, bias=False)
mha_with_bias = MultiHeadAttentionWithBias(512, 8, bias=True)

print(f"Without bias: {sum(p.numel() for p in mha_no_bias.parameters()):,} params")
print(f"With bias:    {sum(p.numel() for p in mha_with_bias.parameters()):,} params")
print(f"Difference:   {sum(p.numel() for p in mha_with_bias.parameters()) - sum(p.numel() for p in mha_no_bias.parameters()):,} params")

### 8.2 Fused QKV Projection (Efficiency Trick)

In [ ]:
class MultiHeadAttentionFused(nn.Module):
    """
    Multi-Head Attention with fused QKV projection.
    
    Instead of 3 separate linear layers, use one larger one.
    This is more efficient on GPUs due to better memory access patterns.
    """
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Fused QKV projection: one matrix multiply instead of three!
        self.W_QKV = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """Self-attention only (query=key=value)"""
        batch_size, seq_len, _ = x.shape
        
        # Fused projection: (batch, seq_len, d_model) → (batch, seq_len, 3*d_model)
        qkv = self.W_QKV(x)
        
        # Split into Q, K, V
        qkv = qkv.view(batch_size, seq_len, 3, self.num_heads, self.d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, batch, heads, seq_len, d_k)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        attention_output, attention_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )
        
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len, self.d_model)
        
        return self.W_O(attention_output), attention_weights

# Test fused version
mha_fused = MultiHeadAttentionFused(512, 8)
x = torch.randn(2, 10, 512)
output, attn = mha_fused(x)
print(f"Fused MHA output shape: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in mha_fused.parameters()):,}")

---
## Exercises

### Exercise 1: Implement Multi-Head Cross-Attention

Modify the MultiHeadAttention class to support cross-attention where Q comes from one sequence and K,V from another.

In [ ]:
# TODO: Implement multi-head cross-attention
# Hint: The existing implementation already supports this!
# Just pass different tensors for query vs key/value

# Example usage:
# encoder_output = torch.randn(batch, src_len, d_model)  # From encoder
# decoder_state = torch.randn(batch, tgt_len, d_model)   # Current decoder state
# 
# cross_attn = MultiHeadAttention(d_model, num_heads)
# output, weights = cross_attn(
#     query=decoder_state,     # Q from decoder
#     key=encoder_output,      # K from encoder
#     value=encoder_output     # V from encoder
# )

### Exercise 2: Head Importance Analysis

Compute which heads are most "important" by measuring the variance of their attention patterns.

In [ ]:
# TODO: Implement head importance scoring
# Hint: Heads with uniform attention (low variance) may be less useful
#       Heads with focused attention (high variance) may be more important

def compute_head_importance(attention_weights):
    """
    Compute importance score for each head based on attention pattern variance.
    
    Args:
        attention_weights: (batch, num_heads, seq_len, seq_len)
    
    Returns:
        importance: (num_heads,) - higher = more important
    """
    # Your implementation here
    pass

### Exercise 3: Implement Relative Position Attention

As a preview of the next topic (Positional Encoding), try adding relative position information to the attention scores.

In [ ]:
# TODO: Add relative position bias to attention
# The idea: tokens that are closer together should have a position-based bias

def relative_position_bias(seq_len):
    """
    Create a relative position bias matrix.
    
    Position i attending to position j has bias based on (i - j)
    
    Returns:
        bias: (seq_len, seq_len)
    """
    # Your implementation here
    pass

---
## Summary

### What We Learned

1. **Why Multi-Head**: Single attention can only capture one relationship type; multiple heads capture different patterns simultaneously

2. **The Formula**:
   $$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

3. **Efficient Implementation**: One large projection + reshape is faster than separate projections

4. **Same Parameters**: More heads doesn't mean more parameters!

5. **Causal Masking**: Works the same way, just add a mask

### Key Takeaways

| Aspect | Single Head | Multi-Head |
|--------|-------------|------------|
| Expressiveness | One pattern | Multiple patterns |
| Parameters | 4 × d² | 4 × d² (same!) |
| Dim per head | d | d/h |
| Interpretation | Harder | Easier (can analyze each head) |

### Next Up: Positional Encoding

Attention has no notion of position - "cat sat mat" and "mat sat cat" would look the same! Next, we'll learn how to add position information.

---
## Bonus: Attention Visualization Helper

In [ ]:
def visualize_attention(attention_weights, tokens, title="Attention Visualization"):
    """
    Utility function to visualize multi-head attention.
    
    Args:
        attention_weights: (batch, num_heads, seq_len, seq_len)
        tokens: List of token strings
        title: Plot title
    """
    num_heads = attention_weights.size(1)
    seq_len = len(tokens)
    
    # Calculate grid dimensions
    cols = min(4, num_heads)
    rows = (num_heads + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    if num_heads == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    
    for head_idx in range(num_heads):
        row, col = head_idx // cols, head_idx % cols
        ax = axes[row][col]
        
        attn = attention_weights[0, head_idx].detach().cpu().numpy()
        im = ax.imshow(attn, cmap='viridis', vmin=0, vmax=1)
        ax.set_title(f'Head {head_idx + 1}', fontsize=10)
        ax.set_xticks(range(seq_len))
        ax.set_yticks(range(seq_len))
        ax.set_xticklabels(tokens, fontsize=8, rotation=45, ha='right')
        ax.set_yticklabels(tokens, fontsize=8)
    
    # Hide empty subplots
    for idx in range(num_heads, rows * cols):
        row, col = idx // cols, idx % cols
        axes[row][col].axis('off')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# Example usage
tokens = ["The", "quick", "brown", "fox", "jumps"]
mha = MultiHeadAttention(64, 4, dropout=0.0)
x = torch.randn(1, 5, 64)
_, attn = mha(x, x, x)
visualize_attention(attn, tokens, "Multi-Head Attention Example")